In [ ]:
import pandas as pd # Librería de lectura de datos

import matplotlib.pyplot as plt # Librería de visualización de datos

from sklearn.model_selection import train_test_split,GridSearchCV # Función para dividir los datos en entrenamiento y prueba
from sklearn.preprocessing import MinMaxScaler,OneHotEncoder,LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier,plot_tree, export_text

from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve # Funciones para evaluar el rendimiento del modelo

from sklearn.feature_selection import (VarianceThreshold, SelectKBest,
                                      f_classif, RFE, SelectFromModel)
from sklearn import svm
from sklearn.metrics import confusion_matrix as cm_func
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import confusion_matrix as cm_func2
from xgboost import XGBClassifier
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.metrics import confusion_matrix as cm_func3

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

df = pd.read_parquet('../clean_data/mushrooms_limpio.parquet')

df.info()
# Resumen del análisis
- **Forma:** (8124, 21)
- **Todas las columnas son categóricas** (tipo `str`).
- **Nulos:** 0 en todas las columnas.
- **Balance de clases:** `e` 51.8%, `p` 48.2% (casi balanceado).
- **Variabilidad:** varias columnas con baja cardinalidad (2–6 valores); algunas con más (hasta 12).

Recomendaciones rápidas:
- Para clustering con algoritmos basados en distancia (KMeans, KNN): convertir categóricas con `OneHotEncoder` (sparse), luego reducir dimensión (TruncatedSVD o UMAP) y aplicar `StandardScaler`.
- Si quieres evitar expansión dimensional: usar codificaciones por frecuencia / target encoding / hashing, o usar `KModes` / `k-prototypes` para datos categóricos.
- No escalar variables para métodos basados en árboles o `KModes`.
- Imputación: `SimpleImputer(strategy='most_frequent')` si aparecen nulos en otras fuentes.

He incluido a continuación un ejemplo de pipeline reproducible que puedes ejecutar (celda 3).

In [8]:
# Cargar el dataset (intenta parquet, si falla usa CSV)
try:
    df = pd.read_parquet('../clean_data/mushrooms_limpio.parquet')
except Exception:
    df = pd.read_csv('../clean_data/mushrooms_limpio.csv')

cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()

preproc = ColumnTransformer([
        ('imp_ohe', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
    ]), cat_cols),
], remainder='drop', sparse_threshold=0)

# Ajusta n_components según dimensionalidad (aquí 30 como ejemplo)
n_components = min(30, max(2, len(cat_cols)))
pipe = Pipeline([
    ('preproc', preproc),
    ('svd', TruncatedSVD(n_components=n_components, random_state=0)),
    ('scaler', StandardScaler()),
    ('km', KMeans(n_clusters=3, random_state=0))
])

pipe.fit(df)
labels = pipe.named_steps['km'].labels_
print('clusters assigned:', len(set(labels)))

# Guarda etiquetas
df['cluster'] = labels
df.to_csv('../clean_data/mushrooms_with_clusters.csv', index=False)
print('Wrote ../clean_data/mushrooms_with_clusters.csv')

clusters assigned: 3
Wrote ../clean_data/mushrooms_with_clusters.csv


## Limpieza avanzada
En esta celda aplicamos: eliminación de duplicados, detección/agrupación de niveles raros en categóricas, imputación (numérica: mediana, categórica: moda) y detección básica de outliers numéricos (IQR).
- Para variables categóricas: se agrupan niveles con frecuencia < 1% en `rare` y luego se imputan con la categoría más frecuente.
- Para variables numéricas: se detectan outliers por IQR (se pueden eliminar o winsorizar manualmente).
- Finalmente se guarda `clean_data/mushrooms_cleaned_advanced.csv`.

In [10]:
# Cargar dataset
try:
    df = pd.read_parquet('../clean_data/mushrooms_limpio.parquet')
except Exception:
    df = pd.read_csv('../clean_data/mushrooms_limpio.csv')

print('Initial shape:', df.shape)

# 1) Duplicados
dups = df.duplicated().sum()
print('duplicates:', dups)
df = df.drop_duplicates()
print('After drop duplicates shape:', df.shape)

# 2) Identificar numéricas / categóricas
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()
print('num_cols:', num_cols)
print('cat_cols len:', len(cat_cols))

# Convertir categóricas 'category' a object para poder añadir niveles nuevos
for c in cat_cols:
    # evite Pandas4Warning usando isinstance sobre el dtype CategoricalDtype
    if isinstance(df[c].dtype, pd.CategoricalDtype):
        df[c] = df[c].astype('object')

# 3) Outliers numéricos (IQR)
outlier_summary = {}
for c in num_cols:
    Q1 = df[c].quantile(0.25)
    Q3 = df[c].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    n_out = ((df[c] < lower) | (df[c] > upper)).sum()
    outlier_summary[c] = int(n_out)
print('numeric outliers per col:', outlier_summary)

# 4) Categóricas: agrupar niveles raros (<1%) como 'rare'
rare_thresh = 0.01
cat_rare_counts = {}
for c in cat_cols:
    freqs = df[c].value_counts(normalize=True)
    rare_levels = freqs[freqs < rare_thresh].index.tolist()
    cat_rare_counts[c] = len(rare_levels)
    if rare_levels:
        df[c] = df[c].where(~df[c].isin(rare_levels), other='rare')
print('rare levels replaced per cat col (count):')
print(cat_rare_counts)

# 5) Imputación: numérica -> mediana, categórica -> moda
if num_cols:
    num_imp = SimpleImputer(strategy='median')
    df[num_cols] = num_imp.fit_transform(df[num_cols])

cat_imp = SimpleImputer(strategy='most_frequent')
if cat_cols:
    df[cat_cols] = cat_imp.fit_transform(df[cat_cols])

print('Nulls after imputation:')
print(df.isnull().sum().sum())

# 6) Resumen y guardado
print('Final shape:', df.shape)
out_path = '../clean_data/mushrooms_cleaned_advanced.csv'
df.to_csv(out_path, index=False)
print('Wrote', out_path)

Initial shape: (8124, 21)
duplicates: 0
After drop duplicates shape: (8124, 21)
num_cols: []
cat_cols len: 21
numeric outliers per col: {}
rare levels replaced per cat col (count):
{'class': 0, 'cap-shape': 2, 'cap-surface': 1, 'cap-color': 3, 'bruises': 0, 'odor': 1, 'gill-attachment': 0, 'gill-spacing': 0, 'gill-size': 0, 'gill-color': 2, 'stalk-shape': 0, 'stalk-surface-above-ring': 1, 'stalk-surface-below-ring': 0, 'stalk-color-above-ring': 2, 'stalk-color-below-ring': 2, 'veil-color': 1, 'ring-number': 1, 'ring-type': 2, 'spore-print-color': 5, 'population': 0, 'habitat': 0}
Nulls after imputation:
0
Final shape: (8124, 21)
Wrote ../clean_data/mushrooms_cleaned_advanced.csv


## Reducción de dimensionalidad: PCA + t-SNE
Aplicamos dos técnicas complementarias:
1. **PCA (Principal Component Analysis):** Reduce preservando 95% de la varianza. Útil para entrenar modelos rápidamente con menos features.
2. **t-SNE (t-Distributed Stochastic Neighbor Embedding):** Reduce a 2D manteniendo la estructura local de los datos. Perfecto para visualización.

Proceso:
- OneHotEncode las columnas categóricas → 113 features
- StandardScaler para normalizar
- PCA con 95% varianza → ~59 componentes
- t-SNE en 2D (después de reducir PCA a 50 dims para eficiencia) → visualización
- Guardar ambas versiones en CSV para usar en modelos posteriores

In [12]:
# Cargar dataset limpio
try:
    df = pd.read_parquet('../clean_data/mushrooms_limpio.parquet')
except Exception:
    df = pd.read_csv('../clean_data/mushrooms_limpio.csv')

print('Original shape:', df.shape)

# Identificar columnas categóricas y numéricas
cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print('cat_cols:', len(cat_cols), '| num_cols:', len(num_cols))

# 1) OneHotEncode + StandardScaler
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
], remainder='passthrough' if num_cols else 'drop')

X = preprocessor.fit_transform(df[cat_cols + num_cols])
print('After OneHot shape:', X.shape)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2) PCA con 95% de varianza
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)
print(f'After PCA (95% var) shape: {X_pca.shape}')
print(f'Explained variance: {pca.explained_variance_ratio_.sum():.4f}')

# 3) t-SNE (primero reducir a 50 dims para eficiencia)
dims_for_tsne = min(50, X_pca.shape[1])
if X_pca.shape[1] > dims_for_tsne:
    pca_reduce = PCA(n_components=dims_for_tsne)
    X_pca_reduced = pca_reduce.fit_transform(X_pca)
else:
    X_pca_reduced = X_pca

print('Input to t-SNE shape:', X_pca_reduced.shape)
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_pca_reduced)
print('After t-SNE shape:', X_tsne.shape)

# 4) Guardar resultados
df_pca = pd.DataFrame(X_pca, columns=[f'pca_{i}' for i in range(X_pca.shape[1])])
df_tsne = pd.DataFrame(X_tsne, columns=['tsne_1', 'tsne_2'])

# Añadir clase original si existe
if 'class' in df.columns:
    df_pca['class'] = df['class'].values
    df_tsne['class'] = df['class'].values

df_pca.to_csv('../clean_data/mushrooms_pca.csv', index=False)
df_tsne.to_csv('../clean_data/mushrooms_tsne.csv', index=False)

print(f'Saved: mushrooms_pca.csv ({df_pca.shape}), mushrooms_tsne.csv ({df_tsne.shape})')

Original shape: (8124, 21)
cat_cols: 21 | num_cols: 0
After OneHot shape: (8124, 113)
After PCA (95% var) shape: (8124, 59)
Explained variance: 0.9518
Input to t-SNE shape: (8124, 50)
After t-SNE shape: (8124, 2)
Saved: mushrooms_pca.csv ((8124, 60)), mushrooms_tsne.csv ((8124, 3))
